# IMU Analysis
This notebook analyses `imu_challenge_dataset_ext.csv` in three sections:
1. **Raw signal visualisation** — every channel plotted vs time
2. **Gyro-only attitude** — dead-reckoning via cumulative integration of ω
3. **Accelerometer-only attitude** — static tilt estimation via `atan2`

## Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_FILE = 'imu_challenge_dataset_ext.csv'
df = pd.read_csv(CSV_FILE, skiprows=1)

TIME_COL = 'time_s'
t = df[TIME_COL].values
data_cols = [c for c in df.columns if c != TIME_COL]

print(f'Samples : {len(df):,}')
print(f'Duration: {t[-1]:.1f} s')
print(f'Columns : {data_cols}')
df.head(3)

---
## Section 1 — Raw Signal Visualisation
Each sensor channel is shown in its own subplot so scales don't interfere.
All subplots share the same time axis.

In [ ]:
n = len(data_cols)
fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)

for ax, col in zip(axes, data_cols):
    ax.plot(t, df[col].values, linewidth=0.7)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('IMU Dataset — All Channels vs Time', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('all_channels.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — Gyro-Only Attitude Estimation
Attitude is obtained by **dead-reckoning**: integrating the angular velocity ω over time.

$$\theta(t) = \int_0^t \omega(\tau)\, d\tau \approx \sum_k \omega_k \, \Delta t_k$$

- **Roll** → integrate `gx` (rotation about X)
- **Pitch** → integrate `gy` (rotation about Y)
- **Yaw** → integrate `gz` (rotation about Z)

> **Limitation:** gyro integration accumulates drift over time (no correction signal).

In [ ]:
dt = np.diff(t, prepend=t[0])  # Δt per sample (first element = 0)

roll_gyro  = np.degrees(np.cumsum(df['gx_rad_s'].values * dt))
pitch_gyro = np.degrees(np.cumsum(df['gy_rad_s'].values * dt))
yaw_gyro   = np.degrees(np.cumsum(df['gz_rad_s'].values * dt))

gyro_angles = [
    ('Roll  (gx)', roll_gyro,  'tab:blue'),
    ('Pitch (gy)', pitch_gyro, 'tab:orange'),
    ('Yaw   (gz)', yaw_gyro,   'tab:green'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, (label, data, color) in zip(axes, gyro_angles):
    ax.plot(t, data, linewidth=0.7, color=color)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('Gyro-Only Attitude (Dead-Reckoning)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gyro_attitude.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3 — Accelerometer-Only Attitude Estimation (atan2)
When the sensor is in quasi-static motion, gravity dominates the accelerometer reading.
Tilt angles can then be recovered geometrically:

$$\text{Roll}  = \mathrm{atan2}(a_y,\, a_z)$$
$$\text{Pitch} = \mathrm{atan2}\!\left(-a_x,\, \sqrt{a_y^2 + a_z^2}\right)$$

> **Yaw** cannot be recovered from the accelerometer alone — gravity is symmetric about the vertical axis.  
> **Limitation:** any linear acceleration contaminates the estimate (not drift-free like gyro, but also not divergent).

In [ ]:
ax_  = df['ax_m_s2'].values
ay_  = df['ay_m_s2'].values
az_  = df['az_m_s2'].values

roll_acc  = np.degrees(np.arctan2(ay_, az_))
pitch_acc = np.degrees(np.arctan2(-ax_, np.sqrt(ay_**2 + az_**2)))

acc_angles = [
    ('Roll  (atan2(ay, az))',              roll_acc,  'tab:blue'),
    ('Pitch (atan2(-ax, sqrt(ay²+az²)))', pitch_acc, 'tab:orange'),
]

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, (label, data, color) in zip(axes, acc_angles):
    ax.plot(t, data, linewidth=0.7, color=color)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Angle (deg)')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('Accelerometer-Only Attitude (atan2 tilt estimation)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('acc_attitude.png', dpi=150, bbox_inches='tight')
plt.show()